# SLURM Magic — Jupyter Notebook Presentation

**Goal:** 10-minute presentation + 2 minutes Q&A on using SLURM magics inside Jupyter.

**Agenda:**
- Install and load `slurm-magic`
- Examples: `%%sbatch`, `%salloc`, `%squeue`
- Interactive patterns and kernel tips
- Converting notebook to slides and sharing

## Install and load slurm-magic

This notebook uses the NERSC `slurm-magic` extension which exposes SLURM commands as IPython magics (e.g. `%%sbatch`, `%salloc`, `%squeue`).

If you used `setup_env.sh` the package is already installed from `requirements.txt`. Otherwise install manually:



In [2]:
from IPython import get_ipython
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

ip = get_ipython()
matches = [path for path in Path.cwd().rglob('slurm_magic.py') if 'p4_slurm_notebook' in path.parts]
if not matches:
    raise FileNotFoundError('Could not find local slurm_magic.py in the workspace tree')
module_path = sorted(matches)[0]
spec = spec_from_file_location('local_slurm_magic', module_path)
module = module_from_spec(spec)
spec.loader.exec_module(module)
module.load_ipython_extension(ip)
print(f'Loaded local slurm_magic from {module_path}')

Loaded local slurm_magic from /net/afscra/people/plgjworek/p4_slurm_notebook/slurm_magic.py


## `%%sbatch` — submit a cell as a batch job

Use `%%sbatch` at the top of a cell to submit its contents as a Slurm batch script. The magic returns the submitted job id and captures output into files named like `slurm-<JOBID>.out`.

Example below submits a tiny job that prints a message. This will only succeed on a system with SLURM and accessible `sbatch`.


In [3]:
%%sbatch -N1 -t 00:02:00 --job-name=notebook-demo
#!/bin/bash
module load python || true
python - <<'PY'
print('Hello from slurm-magic %%sbatch job')
PY

'Submitted batch job 20216271\n'

## `%salloc` and `%srun` — interactive allocations

You can request an allocation and run commands directly. `%salloc` behaves like the command-line `salloc` and will run the given command inside the allocation.

Example (will block until allocation is obtained on a SLURM cluster):

In [4]:
%salloc -N1 -t 00:05:00 -- srun --pty bash -l
# After that command completes you'll have an interactive shell on a compute node (depends on cluster policy)

'salloc: Pending job allocation 20216272\nsalloc: job 20216272 queued and waiting for resources\nsalloc: job 20216272 has been allocated resources\nsalloc: Granted job allocation 20216272\nsrun: error: ioctl(TIOCGWINSZ): Inappropriate ioctl for device\nsrun: error: Not using a pseudo-terminal, disregarding --pty option\nsalloc: Relinquishing job allocation 20216272\nsalloc: Job allocation 20216272 has been revoked.\n'

## `%squeue` — inspect jobs

`%squeue` wraps the `squeue` command. Use the local fix in `slurm_magic.py` so the command returns text cleanly before pandas parsing.

If you want a table output, switch to pandas mode after loading the local extension.

Example: list your queued/running jobs.

In [6]:
#%slurm display raw
%squeue -u $USER

# Optional table output:
# %slurm display pandas
# %squeue -u $USER

'             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)\n\nsqueue: error: Invalid user: $USER\n\n'

## Converting to slides & sharing

- Convert to Reveal.js slides: `jupyter nbconvert --to slides slurm_magic_presentation.ipynb --reveal-prefix "https://cdnjs.cloudflare.com/ajax/libs/reveal.js/3.3.0/"`
- For live slides inside the notebook, install RISE (`pip install RISE`).
- Share `slurm_magic_presentation.ipynb`, `requirements.txt`, and `setup_env.sh` for reproducibility.

---
**End / Q&A**